# 04 · Загвар: LightGBM, XGBoost, CatBoost + Optuna

Гурван gradient boosting алгоритмыг **ижил нөхцөлд** сургаж, аль нь хамгийн сайныг **validation**-аар сонгоно.

| Юу | Тохиргоо (өөрчлөгдөхгүй) |
|---|---|
| Feature | 18 (`feature_list.json`, `category` хассан — ADR 0005) |
| Target | Δ = y_h − y_0 (y = log1p(анхаарал)), h = 1, 2, 3 тус бүрт тусдаа model |
| Loss | L1 (MAE) — медианыг таамагладаг тул вирал үсрэлтэд тэсвэртэй |
| Хуваалт | train ≤ 2025-06-30 < valid ≤ 2026-03-31 < test |

**Дүрэм:** сонголт зөвхөн **(в)-гүй valid MAE**-ээр. **Test-ийг энд огт ашиглахгүй** (дараагийн 05 алхамд нэг удаа).

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import plotly.express as px, plotly.graph_objects as go, plotly.io as pio
from plotly.subplots import make_subplots
import optuna
warnings.filterwarnings("ignore")
pio.templates.default = "plotly_white"
BLUE, RED, GRAY = "#2a78d6", "#e63946", "#9aa0ad"
COLORS = {"lightgbm": BLUE, "xgboost": "#f0a500", "catboost": RED}

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import train
from src.features import HORIZONS

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Сургалт ба сонголтын өгөгдөл

- **Сургалт:** train хуваалтын **2023-07-01-ээс** хойшх мөр (хурдны тулд). **(в) бүлэг орно** (ADR 0009): тэдгээрийн train үеийн мөр нь
  оргилоос өмнөх нам гүм үе тул ирээдүйн мэдээлэл агуулахгүй.
- **Хилийн purge:** t+3 нь хуваалтаас гарсан мөрийг `features.py` аль хэдийн хассан (ADR 0009).
- **Valid (early stopping + сонголт):** **(в)-гүй**. (в) нь valid үед оргилсон учраас сонгогдсон сэдэв, түүнийг оруулбал
  хэт "үсрэлттэй" хугацаан дээр сонголт хийгдэнэ (ADR 0005).

In [2]:
feats = pd.read_parquet(ROOT / "data/processed/features.parquet")
feats = feats[feats["split"] != "test"]                 # test-ийг уншаад л хаяна
groups = train.load_groups()
rs = train.row_sets(feats, groups)
g = feats["article"].map(groups)
pd.DataFrame({
    "сургалт (train ≥ 2023-07-01)": g[rs["train"]].value_counts(),
    "сонголт (valid, (в)-гүй)": g[rs["valid"]].value_counts(),
}).reindex(["а", "б", "в"]).fillna(0).astype(int).rename_axis("бүлэг")

,сургалт (train ≥ 2023-07-01),"сонголт (valid, (в)-гүй)"
бүлэг,,
а,109387,40921
б,78909,30894
в,16990,0


## 2. Optuna: hyperparameter-ийг автоматаар хайх

Hyperparameter (модны гүн, learning rate, regularization...) нь model **өөрөө сурдаггүй**, бид тохируулдаг тоонууд.
Гараар таах оронд **Optuna** ашиглана:

- **TPE sampler** — өмнөх trial-уудаас "аль бүс сайн байсан бэ"-г сурч, дараагийн оролдлогыг тэнд ойртуулна (санамсаргүй хайлтаас ухаалаг).
- **MedianPruner** — trial 10 iteration тутам valid MAE-ээ мэдээлнэ. Өмнөх trial-уудын медианаас муу явж байвал **эрт зогсооно** (цаг хэмнэнэ).
- **Early stopping 100** — valid MAE 100 iteration сайжрахгүй бол зогсож, хамгийн сайн iteration-ыг үлдээнэ.

Алгоритм × horizon бүрт **25 trial** = 3 × 3 × 25 = **225 trial**. h бүрийг **тусад нь** тааруулсан (h1-ийн параметрийг дахин ашиглаагүй).

> ⏱ Энэ cell ~25 минут ажиллана (4 CPU).

In [3]:
res = train.run()
table, studies, winner = res["table"], res["studies"], res["winner"]

train 205,286 мөр (2023-07-01-аас, (в) орсон), valid (в)-гүй 71,815 мөр


lightgbm  h1: valid 0.0944 (naive 0.1078), train 0.0953, 715 iter, 25 trial (16 pruned), 2.0 мин


lightgbm  h2: valid 0.1257 (naive 0.1526), train 0.1261, 772 iter, 25 trial (17 pruned), 1.7 мин


lightgbm  h3: valid 0.1409 (naive 0.1749), train 0.1422, 629 iter, 25 trial (17 pruned), 1.5 мин


xgboost   h1: valid 0.0943 (naive 0.1078), train 0.0919, 1158 iter, 25 trial (14 pruned), 1.7 мин


xgboost   h2: valid 0.1258 (naive 0.1526), train 0.1228, 1103 iter, 25 trial (16 pruned), 1.7 мин


xgboost   h3: valid 0.1411 (naive 0.1749), train 0.1388, 1002 iter, 25 trial (15 pruned), 1.5 мин


catboost  h1: valid 0.0945 (naive 0.1078), train 0.0906, 1496 iter, 25 trial (15 pruned), 4.3 мин


catboost  h2: valid 0.1256 (naive 0.1526), train 0.1227, 1249 iter, 25 trial (16 pruned), 3.6 мин


catboost  h3: valid 0.1408 (naive 0.1749), train 0.1368, 1264 iter, 25 trial (12 pruned), 4.7 мин
Ялагч: catboost


### Study: шилдэг параметрүүд

Алгоритм × horizon бүрийн хамгийн сайн trial (valid MAE нь (в)-гүй):

In [4]:
rows = []
for (algo, h), st in studies.items():
    states = pd.Series([t.state.name for t in st.trials]).value_counts()
    rows.append({"algo": algo, "h": h, "valid MAE": round(st.best_value, 4), "шилдэг trial": st.best_trial.number,
                 "дууссан": int(states.get("COMPLETE", 0)), "pruned": int(states.get("PRUNED", 0)),
                 **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in st.best_params.items()}})
pd.DataFrame(rows).set_index(["algo", "h"])

valid MAE  шилдэг trial  дууссан  pruned  learning_rate  \
algo     h                                                            
lightgbm 1     0.0944             1        9      16         0.0229   
         2     0.1257             1        8      17         0.0229   
         3     0.1409             1        8      17         0.0229   
xgboost  1     0.0943             1       11      14            NaN   
         2     0.1258             1        9      16            NaN   
         3     0.1411             1       10      15            NaN   
catboost 1     0.0945             2       10      15         0.0315   
         2     0.1256             2        9      16         0.0315   
         3     0.1408             2       13      12         0.0315   

            num_leaves  min_data_in_leaf  feature_fraction  bagging_fraction  \
algo     h                                                                     
lightgbm 1       174.0             208.0             0.854            0.5103   
         2       174.0             208.0             0.854            0.5103   
         3       174.0             208.0             0.854            0.5103   
xgboost  1         NaN               NaN               NaN               NaN   
         2         NaN               NaN               NaN               NaN   
         3         NaN               NaN               NaN               NaN   
catboost 1         NaN               NaN               NaN               NaN   
         2         NaN               NaN               NaN               NaN   
         3         NaN               NaN               NaN               NaN   

            lambda_l2     eta  max_depth  min_child_weight  subsample  \
algo     h                                                              
lightgbm 1     7.5795     NaN        NaN               NaN        NaN   
         2     7.5795     NaN        NaN               NaN        NaN   
         3     7.5795     NaN        NaN               NaN        NaN   
xgboost  1        NaN  0.0229        9.0           15.9305     0.8540   
         2        NaN  0.0229        9.0           15.9305     0.8540   
         3        NaN  0.0229        9.0           15.9305     0.8540   
catboost 1        NaN     NaN        NaN               NaN     0.6062   
         2        NaN     NaN        NaN               NaN     0.6062   
         3        NaN     NaN        NaN               NaN     0.6062   

            colsample_bytree  lambda  depth  l2_leaf_reg     rsm  
algo     h                                                        
lightgbm 1               NaN     NaN    NaN          NaN     NaN  
         2               NaN     NaN    NaN          NaN     NaN  
         3               NaN     NaN    NaN          NaN     NaN  
xgboost  1            0.5103  7.5795    NaN          NaN     NaN  
         2            0.5103  7.5795    NaN          NaN     NaN  
         3            0.5103  7.5795    NaN          NaN     NaN  
catboost 1               NaN     NaN   10.0      16.9675  0.5909  
         2               NaN     NaN   10.0      16.9675  0.5909  
         3               NaN     NaN   10.0      16.9675  0.5909

### Study: ялагчийн h3 trial бүр

Trial бүр ямар параметр туршиж, ямар үр дүн гаргасныг харуулна. `PRUNED` нь MedianPruner эрт зогсоосон trial.

In [5]:
st = studies[(winner, 3)]
df = st.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
df.columns = [c.replace("params_", "").replace("user_attrs_", "") for c in df.columns]
df.round(4)

,number,value,state,depth,l2_leaf_reg,learning_rate,rsm,subsample,best_iteration
0,0,0.1412,COMPLETE,10,12.0571,0.0711,0.5780,0.7993,424.0
1,1,0.1414,COMPLETE,4,19.0304,0.0430,0.8540,0.8006,2994.0
2,2,0.1408,COMPLETE,10,16.9675,0.0315,0.5909,0.6062,1264.0
3,3,0.1410,COMPLETE,6,5.9584,0.0458,0.6456,0.7160,2181.0
4,4,0.1417,COMPLETE,4,2.7011,0.1227,0.7280,0.6832,1224.0
5,5,0.1417,PRUNED,5,5.7489,0.1829,0.5232,0.7962,NaN
6,6,0.1413,COMPLETE,5,1.2476,0.1215,0.9828,0.9744,845.0
7,7,0.1415,COMPLETE,6,1.3940,0.1930,0.7201,0.8421,484.0
8,8,0.1449,PRUNED,7,1.1241,0.0397,0.6294,0.9547,NaN
9,9,0.1414,COMPLETE,6,5.8641,0.1379,0.5924,0.7734,421.0


In [6]:
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{a}" for a in train.ALGOS], shared_yaxes=False)
for i, algo in enumerate(train.ALGOS, 1):
    st = studies[(algo, 3)]
    done = [t for t in st.trials if t.value is not None]
    x, y = [t.number for t in done], [t.value for t in done]
    fig.add_scatter(x=x, y=y, mode="markers", marker_color=GRAY, name="trial", showlegend=i == 1, row=1, col=i)
    fig.add_scatter(x=x, y=np.minimum.accumulate(y), mode="lines", line_color=COLORS[algo], name="хамгийн сайн", showlegend=i == 1, row=1, col=i)
fig.update_layout(title="Optuna-ийн түүх (h3): trial бүрийн valid MAE ба тэр хүртэлх хамгийн сайн", height=380)
fig.update_xaxes(title="trial")
fig.show()

## 3. Сонголт: гурван алгоритмын train / valid MAE

- **valid MAE** = (в)-гүй valid (сонголтын шалгуур). **valid (бүгд)** = (в)-тэй, зөвхөн мэдээлэлд.
- **naive** = "өнөөдрийнх шиг" (Δ̂ = 0). **skill** = 1 − MAE / MAE_naive.
- **train − valid** нь overfit-ийн дохио: train хэт бага, valid өндөр бол model цээжилсэн гэсэн үг.

In [7]:
t = table.copy()
t["skill"] = 1 - t["valid_mae"] / t["naive_valid_mae"]
t["train − valid"] = t["train_mae"] - t["valid_mae"]
show = t.set_index(["algo", "h"])[["train_mae", "valid_mae", "valid_mae_all", "naive_valid_mae", "skill", "train − valid", "best_iteration"]]
show.columns = ["train MAE", "valid MAE", "valid (бүгд)", "naive", "skill", "train − valid", "iteration"]
show.style.format({c: "{:.4f}" for c in show.columns[:4]} | {"skill": "{:.1%}", "train − valid": "{:+.4f}"})

In [8]:
avg = table.groupby("algo")[["train_mae", "valid_mae"]].mean().sort_values("valid_mae")
avg.columns = ["train MAE (дундаж)", "valid MAE (дундаж)"]
print(f"Ялагч (h1–h3-ын дундаж valid MAE хамгийн бага): {winner}")
avg.round(4)

Ялагч (h1–h3-ын дундаж valid MAE хамгийн бага): catboost


,train MAE (дундаж),valid MAE (дундаж)
algo,,
catboost,0.1167,0.1203
lightgbm,0.1212,0.1203
xgboost,0.1178,0.1204


## 4. Overfit: ялагчийн learning curve

Iteration (мод) нэмэгдэх тусам train ба valid MAE хэрхэн өөрчлөгдсөнийг харна.
Valid муудаж эхэлвэл (эсвэл 100 iteration сайжрахгүй бол) early stopping зогсооно — **босоо зураас** нь сонгосон iteration.

In [9]:
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"h{h}" for h in HORIZONS])
for i, h in enumerate(HORIZONS, 1):
    f = res["fitted"][(winner, h)]
    it = np.arange(1, len(f.curve["valid"]) + 1)
    fig.add_scatter(x=it, y=f.curve["train"], name="train", line_color=GRAY, showlegend=i == 1, row=1, col=i)
    fig.add_scatter(x=it, y=f.curve["valid"], name="valid ((в)-гүй)", line_color=COLORS[winner], showlegend=i == 1, row=1, col=i)
    fig.add_vline(x=f.best_iteration, line_dash="dot", line_color=RED, row=1, col=i)
fig.update_layout(title=f"{winner}: learning curve (MAE, Δ)", height=380)
fig.update_xaxes(title="iteration")
fig.show()

## 5. Аль feature хамгийн чухал вэ?

Ялагчийн importance-ийг (gain) h1–h3-аар дундажлав.

In [10]:
imp = pd.DataFrame(res["meta"]["importance_top10"]).sort_values("share")
fig = px.bar(imp, x="share", y="feature", orientation="h", color_discrete_sequence=[COLORS[winner]],
             title=f"{winner}: importance top-10 (h1–h3 дундаж)")
fig.update_layout(xaxis_tickformat=".0%", xaxis_title=None, yaxis_title=None, height=420)
fig.show()

## 6. Экспорт: `app/model/`

Ялагчийн 3 model (`h1`, `h2`, `h3`) + `meta.json` (algo, 18 feature, learning curve, importance top-10, сургасан огноо).
Вэб (ADR 0008) яг **энэ** model-ийг ачаална. Шалгалт: файлаас ачаалсан model ижил таамаг гаргах ёстой.

In [11]:
m = train.load()
va = res["valid"]
for h in HORIZONS:
    same = np.array_equal(m.predict(va, h), train.predict(res["fitted"][(winner, h)], va[res["features"]]))
    print(f"h{h}: {m.meta['files'][f'h{h}']:8s} ачаалсан = сургасан: {same}")
{k: m.meta[k] for k in ["algo", "trained_at", "train_from", "train_rows", "valid_rows", "best_iteration"]}

h1: h1.cbm   ачаалсан = сургасан: True


h2: h2.cbm   ачаалсан = сургасан: True
h3: h3.cbm   ачаалсан = сургасан: True


{'algo': 'catboost',
 'trained_at': '2026-11-09T19:09:10+00:00',
 'train_from': '2023-07-01',
 'train_rows': 205286,
 'valid_rows': 71815,
 'best_iteration': {'h1': 1496, 'h2': 1249, 'h3': 1264}}

## 7. 🔥 босго T (ADR 0004)

🔥 нь **хэмжигдэх сэдвийн таамагласан pct(h3) ≥ T** үед тавигдана. T-г **(в)-гүй valid** дээр (сонголттой ижил өгөгдөл) сонгоно:

- Зорилго: өдөрт дунджаар **~5–10 🔥** (дунд нь 7.5). Тааруулсан босго = хэмжигдэх сэдвийн таамгуудаас 7.5 × өдрийн тоо дахь хамгийн өндөр нь.
- Гэхдээ **T = max(тааруулсан, 25%)**: +25%-иас бага өсөлтийг 🔥 гэж нэрлэхгүй (үнэн trend ↑-тэй ижил хил).
- Model-ийг дахин сургахгүй: `app/model/`-д экспортлосон ялагчийг ачаалж, `src/calibrate.py` нь T ба интервалыг `meta.json`-д нэмнэ.

In [12]:
from src import calibrate, scoring
cal = calibrate.run()
va, fire, interval = cal["valid"], cal["fire"], cal["interval"]
meas = va[va["measurable"]]
print(f"(в)-гүй valid: {len(va):,} мөр, {fire['valid_days']} өдөр; хэмжигдэх сэдэв {len(meas):,} мөр ({len(meas)/len(va):.0%})")

T = 25%: өдөрт 3.76 🔥 (T = 25% үед 3.76), 271 өдөр
h1: 80% интервал Δ̂ + [-0.1042, +0.1517]
h2: 80% интервал Δ̂ + [-0.1344, +0.2169]
h3: 80% интервал Δ̂ + [-0.1491, +0.2464]
(в)-гүй valid: 71,815 мөр, 271 өдөр; хэмжигдэх сэдэв 63,413 мөр (88%)


### Таамагласан pct(h3)-ийн тархалт (хэмжигдэх сэдэв)

L1 loss медианыг таамагладаг тул таамаг ихэвчлэн 0-ийн орчим, "болгоомжтой" байдаг. 🔥 нь тархалтын **баруун сүүл**.

In [13]:
q = pd.Series(fire["pred_pct_quantiles"]).rename("таамагласан pct(h3)")
q.index = [f"{float(i):.0%}" for i in q.index]
q.rename_axis("квантил").to_frame().T.style.format("{:+.1%}")

квантил,1%,5%,10%,25%,50%,75%,90%,95%,99%
таамагласан pct(h3),-58.8%,-30.7%,-20.6%,-9.1%,-1.6%,+4.5%,+10.7%,+15.6%,+29.1%


In [14]:
grid = np.round(np.arange(0.05, 1.001, 0.01), 2)
per_day = [scoring.fires_per_day(va, t) for t in grid]
fig = go.Figure()
fig.add_hrect(y0=5, y1=10, fillcolor=GRAY, opacity=0.15, line_width=0, annotation_text="зорилго 5–10", annotation_position="top right")
fig.add_scatter(x=grid, y=per_day, mode="lines", line_color=BLUE, name="өдөрт 🔥 (дундаж)")
fig.add_vrect(x0=grid[0], x1=scoring.T_FLOOR, fillcolor=RED, opacity=0.06, line_width=0)
fig.add_vline(x=fire["T"], line_dash="dot", line_color=RED, annotation_text=f"T = {fire['T']:.0%}")
fig.update_layout(title="Босго T-ээс хамаарч өдөрт дунджаар хэдэн 🔥 гарах вэ (valid, (в)-гүй, хэмжигдэх сэдэв)",
                  xaxis_title="T (таамагласан pct(h3))", yaxis_title="өдөрт 🔥", xaxis_tickformat=".0%", height=400, showlegend=False)
fig.show()
print(f"T = 25% үед өдөрт дунджаар {fire['per_day_at_25pct']} 🔥.  Сонгосон T = {fire['T']:.0%} -> өдөрт {fire['per_day_at_T']} 🔥")

T = 25% үед өдөрт дунджаар 3.76 🔥.  Сонгосон T = 25% -> өдөрт 3.76 🔥


In [15]:
daily = (meas["pred_pct"] >= fire["T"]).groupby(meas["date"]).sum()
fig = px.histogram(daily, nbins=int(daily.max()) + 1, color_discrete_sequence=[RED],
                   title=f"T = {fire['T']:.0%} үед өдөр бүрийн 🔥-ийн тоо ({len(daily)} өдөр)")
fig.update_layout(xaxis_title="тухайн өдрийн 🔥", yaxis_title="өдрийн тоо", showlegend=False, bargap=0.05, height=360)
fig.show()
print(f"🔥-гүй (нам гүм) өдөр: {(daily == 0).mean():.0%},  медиан {daily.median():.0f},  хамгийн их {daily.max()}")

🔥-гүй (нам гүм) өдөр: 38%,  медиан 1,  хамгийн их 34


## 8. 80% интервал (ADR 0003)

Residual r = Δ − Δ̂ (лог хуваарьт)-ийн **10% ба 90%-ийн квантилийг** h бүрт **(в)-гүй valid-ийн бүх мөр** дээр авна
(сонголт ба T-тэй ижил өгөгдөл). Интервал нь **Δ̂ + [q10, q90]**, тэгээд pct руу expm1-ээр хувиргана
(квантил монотон хувиргалтад хадгалагддаг). Valid дээр хамралт 80% нь тодорхойлолтоороо; **жинхэнэ шалгалт нь test** (05).

Жишээ багана: **x_0 = 10** (дунд зэргийн сэдэв), Δ̂ = 0 үед интервал pct-ээр хэд болох вэ.

In [16]:
rows = []
for h in HORIZONS:
    lo, hi = interval[f"h{h}"]["lo"], interval[f"h{h}"]["hi"]
    r = va[f"delta_h{h}"] - va[f"delta_hat_h{h}"]
    p_lo, p_hi = scoring.interval_pct(np.log1p(10.0), 0.0, lo, hi)
    rows.append({"h": f"h{h}", "q10 (Δ)": lo, "q90 (Δ)": hi, "хамралт (valid)": ((r >= lo) & (r <= hi)).mean(),
                 "x_0 = 10, Δ̂ = 0: доод": p_lo, "дээд": p_hi})
pd.DataFrame(rows).set_index("h").style.format({"q10 (Δ)": "{:+.4f}", "q90 (Δ)": "{:+.4f}", "хамралт (valid)": "{:.1%}",
                                                 "x_0 = 10, Δ̂ = 0: доод": "{:+.0%}", "дээд": "{:+.0%}"})

,q10 (Δ),q90 (Δ),хамралт (valid),"x_0 = 10, Δ̂ = 0: доод",дээд
h,,,,,
h1,-0.1042,+0.1517,80.0%,-11%,+18%
h2,-0.1344,+0.2169,80.0%,-14%,+27%
h3,-0.1491,+0.2464,80.0%,-15%,+31%


In [17]:
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"h{h}" for h in HORIZONS], shared_yaxes=True)
for i, h in enumerate(HORIZONS, 1):
    r = (va[f"delta_h{h}"] - va[f"delta_hat_h{h}"]).clip(-1, 1)
    fig.add_histogram(x=r, nbinsx=100, marker_color=GRAY, showlegend=False, row=1, col=i)
    for v in (interval[f"h{h}"]["lo"], interval[f"h{h}"]["hi"]):
        fig.add_vline(x=v, line_dash="dot", line_color=RED, row=1, col=i)
fig.update_layout(title="Residual Δ − Δ̂ (valid, (в)-гүй, ±1-ээр тасалсан): улаан = 10% ба 90%-ийн квантил", height=360)
fig.show()

## ✅ Дүгнэлт

- **Ялагч: CatBoost** — h1–h3-ын дундаж valid MAE ((в)-гүй) **0.1203**. Гэхдээ **LightGBM-тэй бараг тэнцүү** (зөрүү < 0.00001),
  XGBoost 0.1204. Урьдчилан тогтоосон дүрмээр (хамгийн бага дундаж valid MAE) CatBoost сонгогдсон. Бодит байдал дээр гурвуулаа
  **ижил түвшинд** ажиллаж байна: асуудлын хүндрэл нь алгоритмаас биш, өгөгдлөөс (вирал үсрэлтийг урьдчилж мэдэх боломжгүй) хамаарч байна.
- **Naive-аас ~16.5% бага алдаатай** (valid, h бүрт 12–20%). Энэ бол valid дээрх дүн; эцсийн дүгнэлтийг test дээр (а) бүлгээр гаргана.
- **Overfit бага:** CatBoost-ын train − valid зөрүү дунджаар ~0.004 (MAE-ийн ~3%). Learning curve дээр valid MAE early stopping-оос өмнө
  тогтворжиж, дахин өсөөгүй. LightGBM-д зөрүү бараг 0.
- **Optuna:** 225 trial (138 нь MedianPruner-ээр эрт зогссон). Үр дүнгийн "ландшафт" маш **хавтгай**: h3-ын дууссан trial-ууд бүгд
  0.1408–0.1417 хооронд. Тиймээс ихэнх тохиолдолд эхний санамсаргүй (startup) trial хамгийн сайн гарсан, TPE үүнийг давж чадаагүй.
  Seed тогтмол (42) тул h1/h2/h3-т ижил trial-ууд туршигдаж, **ижил параметр** сонгогдсон. Энэ нь h1-ийг дахин ашигласан гэсэн үг **биш**, h бүрийг тусад нь тааруулсан.
- **Сул тал:** MedianPruner бага learning rate-тэй (≈0.03–0.05) trial-уудыг эрт зогсоох хандлагатай, учир нь тэд эхэндээ удаан сайжирдаг.
  Хохирол нь хамгийн ихдээ ~0.0002 MAE гэж тооцоолж байна.
- **Хамгийн чухал feature:** `ratio_7` (өнөөдөр 7 хоногийн дунджаасаа хэр их вэ), `roll_std_3` (сүүлийн өдрүүдийн хэлбэлзэл), `growth_1`, `vs_level`, `dow`.
- Экспорт: `app/model/h1.cbm`, `h2.cbm`, `h3.cbm` + `meta.json`. Файлаас ачаалсан model сургасантай яг ижил таамаг гаргадаг.
- **🔥 босго T = 25%** (доод хил). Valid ((в)-гүй, хэмжигдэх сэдэв) дээр T = 25% үед өдөрт дунджаар **3.76 🔥** гарч байна:
  зорилго ~5–10-аас **цөөн**. 7.5 🔥 гаргахын тулд босгыг 25%-иас доош буулгах шаардлагатай байсан тул дүрмээр (T = max(тааруулсан, 25%))
  25% хэвээр үлдсэн. Шалтгаан нь L1 медиан таамаг "болгоомжтой": таамагласан pct(h3)-ийн 99%-ийн квантил ердөө +29%.
  Өдрүүдийн **38% нь 🔥-гүй** (нам гүм), медиан 1, хамгийн их 34 (ADR 0004-оор энэ нь зөв).
- **80% интервал** (residual-ийн 10%/90% квантил, (в)-гүй valid): h1 Δ̂ + [−0.104, +0.152], h2 [−0.134, +0.217], h3 [−0.149, +0.246].
  Тэгш хэмгүй: дээд тал нь өргөн, учир нь гэнэтийн өсөлт уналтаас илүү огцом. x_0 = 10, Δ̂ = 0 сэдэвт h3-ын интервал ≈ **−15% … +31%**.
- T ба интервал `app/model/meta.json`-ийн `fire`, `interval` талбарт (`python -m src.calibrate`).

**Дараа нь (05):** test-ийг **нэг удаа** (а)/(б)/(в) бүлгээр, 80% интервалын бодит хамралт ч мөн.